# 04. Aya-23 vs Llama-3.1 Tokenizer Subword Fertility Analysis

**Requires GPU + Hub access.** This notebook downloads/loads 8B-parameter models (Aya-23-8B, and/or Llama-3.1-8B which additionally requires accepting Meta's license on the HuggingFace Hub). Run on Colab with a GPU runtime -- see the setup cell below, which auto-clones the repo when a Colab GPU is detected.

In [ ]:
# ============================================================
# PATH BOOSTER — Guarantees project root in sys.path & CWD
# ============================================================
import os, sys
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
proj_dir = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


In [ ]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [ ]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from src.master_corpus.manager import MasterCorpusManager
from src.tokenizer.compare import TokenizerComparator
from src.utils.constants import SUPPORTED_LANGUAGES

manager = MasterCorpusManager()
df = manager.load_sentence_corpus().sample(500, random_state=42)
language_sentences = {lang: df[lang].dropna().astype(str).tolist() for lang in SUPPORTED_LANGUAGES}

## Fertility comparison

In [ ]:
comparator = TokenizerComparator()
fertility_df = comparator.compare(language_sentences)
fertility_df

In [ ]:
relative = comparator.compare(language_sentences)
recommendation = comparator.recommend_base_model(fertility_df, target_language='Ekegusii')
print(f'Recommended base model for Ekegusii: {recommendation}')

## Vocabulary coverage comparison

In [ ]:
comparator.compare_vocabulary_coverage(language_sentences)

## Visualization

In [ ]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from pathlib import Path
from src.visualization.tokenizer import plot_fertility_comparison, plot_vocabulary_coverage

Path('outputs/figures').mkdir(parents=True, exist_ok=True)
plot_fertility_comparison(fertility_df, output_path='outputs/figures/04_fertility.png')
coverage_df = comparator.compare_vocabulary_coverage(language_sentences)
plot_vocabulary_coverage(coverage_df, output_path='outputs/figures/04_vocab_coverage.png')